# 03 — NLP: language, translation, keywords
Tests `roche_poc.nlp`. The translation step uses the DeepL cloud API and consumes quota (free plan: 500 000 characters/month), so it is guarded by a flag and cached after each batch.

Set `DEEPL_API_KEY` in `.env` (never in a file that is committed).

In [ ]:
# Works after `pip install -e .`; the fallback below covers a fresh clone.
import sys, pathlib
_src = pathlib.Path.cwd().resolve().parent / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
import pandas as pd
pd.set_option("display.max_columns", 30, "display.width", 160)

In [ ]:
from roche_poc import config
from roche_poc.data.build_tables import load_table, save_table
from roche_poc.nlp.language import add_language_column
comments = load_table(config.COMMENTS_PATH)
comments = add_language_column(comments)
comments[config.COL_LANGUAGE].value_counts()

## Quota estimate (before spending anything)

In [ ]:
german = comments[comments[config.COL_LANGUAGE] == 'de']
print(len(german), 'German comments,', german[config.COL_COMMENT].str.len().sum(), 'characters to translate')

## Translation (set the flag once you accepted the quota cost)

In [ ]:
RUN_TRANSLATION = False
if RUN_TRANSLATION:
    from roche_poc.nlp.translation import enrich_comments
    comments = enrich_comments(comments)
    save_table(comments, config.COMMENTS_PATH)
    print('Saved', config.COMMENTS_PATH)

## Keywords that distinguish each root cause (TF-IDF)

In [ ]:
from roche_poc.nlp.keywords import top_keywords_by_root_cause
text_col = config.COL_COMMENT_EN if config.COL_COMMENT_EN in comments else config.COL_COMMENT
for cause, words in top_keywords_by_root_cause(comments, text_col=text_col).items():
    print(f'{cause:40s}', ', '.join(w for w, _ in words))

## Next: sub-themes
`roche_poc/nlp/themes.py` is empty on purpose: embeddings + clustering inside one root cause, once the statistics layer is validated.